In [2]:
import polars as pl
from datetime import date, timedelta

## Ranker

In [3]:
user_actions_full = pl.read_parquet('../data/user_actions_full')

In [4]:
ranker_data = pl.read_parquet('../data/user_actions_7_days_ranker')
ranker_data.select(pl.max('date').alias('max_date'), pl.min('date').alias('min_date'))

max_date,min_date
date,date
2024-06-30,2024-06-23


In [5]:
TEST_START = date(2024, 6, 23)

In [6]:
ranker_data

user_id,product_id,date,action_type
i32,i64,date,str
4453096,621177276,2024-06-27,"""view"""
2346229,390557159,2024-06-23,"""view"""
9589535,146396320,2024-06-23,"""view"""
2085760,149293540,2024-06-30,"""view"""
1289097,851213293,2024-06-27,"""view"""
…,…,…,…
9786236,138860210,2024-06-28,"""order"""
6661721,147740723,2024-06-30,"""view"""
8865354,261366874,2024-06-30,"""view"""


In [7]:
ranker_data.group_by('action_type').agg(pl.count("product_id"))

action_type,product_id
str,u32
"""view""",24461070
"""click""",3240738
"""order""",2103547
"""to_cart""",3912035
"""favorite""",193474


In [8]:
map_target = {
    'view': 0,
    'click': 0.01,
    'favorite': 0.1,
    'to_cart': 0.3,
    'order': 0.59,
}
map_target_df = pl.DataFrame(data={
    'action_type': map_target.keys(),
    'target': map_target.values(),
}, schema={"action_type": pl.String, "target": pl.Float32})

In [9]:
# create targets
ranker_data_target = (
    ranker_data
    .join(map_target_df, on='action_type')
    .select('user_id', 'product_id', 'target')
)

In [10]:
ranker_data_target

user_id,product_id,target
i32,i64,f32
4453096,621177276,0.0
2346229,390557159,0.0
9589535,146396320,0.0
2085760,149293540,0.0
1289097,851213293,0.0
…,…,…
9786236,138860210,0.59
6661721,147740723,0.0
8865354,261366874,0.0


In [11]:
ranker_data_target.filter(pl.col('user_id') == 4453096)

user_id,product_id,target
i32,i64,f32
4453096,621177276,0.0
4453096,623810806,0.01
4453096,737735420,0.0
4453096,1539018245,0.0
4453096,496425958,0.0
…,…,…
4453096,570877944,0.0
4453096,1545870731,0.59
4453096,728871749,0.0


In [12]:
pos_users = (
    ranker_data_target
    .group_by('user_id')
    .agg(pl.max('target').alias('max_target'))
    .filter(pl.col('max_target') > 0)
    .sort(by='user_id')
    .sample(100_000, seed=0)
)

In [13]:
ranker_data_target_filtered = (
    ranker_data_target
    .join(pos_users, on='user_id')
    .group_by('user_id', 'product_id')
    .agg(pl.sum('target').alias('target'))
)
ranker_data_target_filtered.shape

(2706980, 3)

In [14]:
ranker_data_target_filtered.sort('user_id')

user_id,product_id,target
i32,i64,f32
54,147887697,0.01
128,1562705601,0.0
128,1023814617,0.0
128,149724906,0.89
170,286030186,0.0
…,…,…
11184165,142624218,0.0
11184165,148481523,0.89
11184165,136780563,0.89


In [15]:
ranker_data_target.filter(pl.col('user_id') == 128)

user_id,product_id,target
i32,i64,f32
128,1023814617,0.0
128,149724906,0.3
128,1562705601,0.0
128,1023814617,0.0
128,149724906,0.59


In [57]:
TEST_START

datetime.date(2024, 6, 23)

In [16]:
data = (
    user_actions_full
    .filter(pl.col('date') < TEST_START)
    .filter(pl.col('date') >= TEST_START - timedelta(days=3 * 30))
)
data.shape   

(102174245, 4)

In [17]:
product_information_full = pl.read_parquet('../data/product_information_full')
product_information_full

product_id,name,brand,type,category_id,category_name
i64,str,str,str,i32,str
160839072,"""CeraVe Смягчающий крем для сух…","""CeraVe""","""Гель для ухода за кожей""",38,"""Сыворотки для лица"""
161689127,"""Yves Rocher / Ив Роше / Увлажн…","""Yves Rocher France""","""Гель для ухода за кожей""",38,"""Сыворотки для лица"""
221508445,"""Bioderma Эликсир для ухода за …","""Bioderma""","""Эликсир для ухода за кожей""",38,"""Сыворотки для лица"""
309017861,"""ART&FACT. / Сыворотка для лица…","""ART&FACT.""","""Сыворотка для лица""",38,"""Сыворотки для лица"""
793710195,"""Breylee Сыворотка для лица Ант…","""Breylee""","""Сыворотка для лица""",38,"""Сыворотки для лица"""
…,…,…,…,…,…
1154315599,"""Сушилка для овощей и фруктов 3…","""Великие реки""","""Дегидратор""",284,"""Сушилки для овощей"""
1196912369,"""GFGRIL Электрическая сушилка д…","""GFGRIL""","""Дегидратор""",284,"""Сушилки для овощей"""
1255681315,"""Дегидратор сушилка для овощей …","""Marta""","""Дегидратор""",284,"""Сушилки для овощей"""


In [18]:
feature_dfs = {}

In [19]:
for suf in ['click', 'favorite', 'to_cart', 'order']:
    feature_dfs[f'{suf}_ui_features'] = (
        data
        .filter(pl.col('action_type') == suf)
        .group_by('user_id', 'product_id')
        .agg(
            pl.count('product_id').alias(f'ui_num_{suf}')
        )
    )
    feature_dfs[f'{suf}_i_features'] = (
        data
        .filter(pl.col('action_type') == suf)
        .group_by('product_id')
        .agg(
            pl.count('user_id').alias(f'i_num_{suf}')
        )
    )

In [20]:
ranker_data_target_filtered_with_features = ranker_data_target_filtered
for key, df in feature_dfs.items():
    if 'ui' in key:
        ranker_data_target_filtered_with_features = (
            ranker_data_target_filtered_with_features
            .join(df, on=['user_id', 'product_id'], how='left')
        )
    else:
        ranker_data_target_filtered_with_features = (
            ranker_data_target_filtered_with_features
            .join(df, on=['product_id'], how='left')
        )

In [21]:
del feature_dfs

In [22]:
del data

In [23]:
ranker_data_target_filtered_with_features.shape

(2706980, 11)

In [24]:
ranker_data_target_filtered_with_features

user_id,product_id,target,ui_num_click,i_num_click,ui_num_favorite,i_num_favorite,ui_num_to_cart,i_num_to_cart,ui_num_order,i_num_order
i32,i64,f32,u32,u32,u32,u32,u32,u32,u32,u32
4595409,1117085974,0.0,null,5022,null,212,null,10575,null,3584
506106,138860216,0.0,null,3971,null,431,null,9858,null,2494
3150910,676294035,0.0,null,8375,null,271,null,5986,null,1418
1889942,256287734,0.0,null,548,null,22,null,813,null,211
11053276,755277238,0.0,null,2261,null,115,null,1028,null,316
…,…,…,…,…,…,…,…,…,…,…
4148921,146885595,0.0,null,3031,null,267,null,14428,null,5415
39571,618166950,0.0,null,989,null,18,null,703,null,170
10423020,164313046,0.0,null,342,null,21,null,424,null,116


In [25]:
ranker_data_target_filtered_with_features = (
    ranker_data_target_filtered_with_features
    .join(
        product_information_full
        .select('product_id', 'brand', 'type', 'category_id'),
        on=['product_id'],
        # how='left'
    )
    # .with_columns(
    #     pl.col('brand').fill_nan(pl.lit('no_brand')),
    #     pl.col('category_id').fill_nan(pl.lit(0)),
    #     pl.col('type').fill_nan(pl.lit('no_type')),
    # )
)

In [26]:
ranker_data_target_filtered_with_features.shape

(2703911, 14)

In [27]:
ranker_data_target_filtered_with_features

user_id,product_id,target,ui_num_click,i_num_click,ui_num_favorite,i_num_favorite,ui_num_to_cart,i_num_to_cart,ui_num_order,i_num_order,brand,type,category_id
i32,i64,f32,u32,u32,u32,u32,u32,u32,u32,u32,str,str,i32
4595409,1117085974,0.0,null,5022,null,212,null,10575,null,3584,"""KIX""","""Туалетная бумага""",399
506106,138860216,0.0,null,3971,null,431,null,9858,null,2494,"""Kinder""","""Печенье""",829
3150910,676294035,0.0,null,8375,null,271,null,5986,null,1418,"""Paclan""","""Перчатки хозяйственные""",576
1889942,256287734,0.0,null,548,null,22,null,813,null,211,"""Азбука Вкуса""","""Салат""",532
11053276,755277238,0.0,null,2261,null,115,null,1028,null,316,"""Фили Бейкер""","""Торт""",308
…,…,…,…,…,…,…,…,…,…,…,…,…,…
4148921,146885595,0.0,null,3031,null,267,null,14428,null,5415,"""Чудо""","""Творожный продукт""",713
39571,618166950,0.0,null,989,null,18,null,703,null,170,"""Весёлая затея""","""Одноразовая посуда для праздни…",245
10423020,164313046,0.0,null,342,null,21,null,424,null,116,"""Dobb&Mopp""","""Губка""",892


In [28]:
import catboost

In [29]:
df = ranker_data_target_filtered_with_features.sort(by='user_id').to_pandas()
mask = df.user_id % 10 <= 7

In [30]:
df.columns

Index(['user_id', 'product_id', 'target', 'ui_num_click', 'i_num_click',
       'ui_num_favorite', 'i_num_favorite', 'ui_num_to_cart', 'i_num_to_cart',
       'ui_num_order', 'i_num_order', 'brand', 'type', 'category_id'],
      dtype='object')

In [31]:
cols = [
    'ui_num_click', 'i_num_click',
    'ui_num_favorite', 'i_num_favorite', 'ui_num_to_cart', 'i_num_to_cart',
    'ui_num_order', 'i_num_order',
]

In [32]:
train_pool = catboost.Pool(
    df.loc[mask, cols],
    label=df.loc[mask].target,
    group_id=df.loc[mask].user_id,
    # cat_features=['brand', 'type', 'category_id'],
)
eval_pool = catboost.Pool(
    df.loc[~mask, cols],
    label=df.loc[~mask].target,
    group_id=df.loc[~mask].user_id,
    # cat_features=['brand', 'type', 'category_id'],
)

In [33]:
params = {
    'iterations': 200,
    'thread_count': -1,
    'depth': 6, 
    'learning_rate': 0.1, 
    'random_state': 1,
    'loss_function': 'YetiRankPairwise',
    'eval_metric': 'NDCG',
#     'eval_metric': 'AUC',
#     'loss_function': 'Logloss',
    'task_type': 'CPU',
}

In [34]:
model = catboost.CatBoost(params)
model.fit(
    train_pool, 
    eval_set=eval_pool,
    use_best_model=True,
    verbose=10,
    early_stopping_rounds=50,
)

0:	test: 0.6482530	best: 0.6482530 (0)	total: 1.5s	remaining: 4m 58s
10:	test: 0.6688431	best: 0.6688431 (10)	total: 15s	remaining: 4m 17s
20:	test: 0.6847555	best: 0.6847555 (20)	total: 28.5s	remaining: 4m 2s
30:	test: 0.6935646	best: 0.6935646 (30)	total: 41.7s	remaining: 3m 47s
40:	test: 0.6974062	best: 0.6975127 (39)	total: 55.6s	remaining: 3m 35s
50:	test: 0.6999962	best: 0.6999962 (50)	total: 1m 9s	remaining: 3m 22s
60:	test: 0.7007403	best: 0.7007403 (60)	total: 1m 23s	remaining: 3m 9s
70:	test: 0.7016842	best: 0.7017383 (69)	total: 1m 36s	remaining: 2m 55s
80:	test: 0.7021263	best: 0.7021263 (80)	total: 1m 50s	remaining: 2m 42s
90:	test: 0.7026110	best: 0.7026475 (89)	total: 2m 10s	remaining: 2m 35s
100:	test: 0.7045530	best: 0.7045530 (100)	total: 2m 25s	remaining: 2m 22s
110:	test: 0.7052143	best: 0.7052544 (109)	total: 2m 39s	remaining: 2m 7s
120:	test: 0.7053066	best: 0.7053066 (120)	total: 2m 53s	remaining: 1m 53s
130:	test: 0.7055647	best: 0.7055647 (130)	total: 3m 7s	rem

In [35]:
name = 'ranker_v2'
model.save_model(f"../models/{name}.bin")

In [36]:
fi = model.get_feature_importance(eval_pool, prettified=True)
fi.head(50)

,Feature Id,Importances
0,ui_num_to_cart,0.016289
1,ui_num_order,0.003838
2,ui_num_click,0.003459
3,i_num_favorite,0.002186
4,ui_num_favorite,0.000361
5,i_num_click,0.000220
6,i_num_to_cart,-0.000023
7,i_num_order,-0.003443


In [77]:
model

In [37]:
model_load = catboost.CatBoost()
model_load.load_model(f"../models/{name}.bin")

In [40]:
import numpy as np

In [52]:
candidates_df_pd = df.loc[~mask]

In [53]:
candidates_df_pd

,user_id,product_id,target,ui_num_click,i_num_click,ui_num_favorite,i_num_favorite,ui_num_to_cart,i_num_to_cart,ui_num_order,i_num_order,brand,type,category_id
1,128,1562705601,0.00,NaN,20.0,NaN,NaN,NaN,1.0,NaN,NaN,YokoSun,Подгузники-трусики,685
2,128,1023814617,0.00,2.0,8415.0,1.0,349.0,1.0,4693.0,1.0,1232.0,SNAQ FABRIQ,Спортивное питание,730
3,128,149724906,0.89,NaN,2642.0,NaN,61.0,1.0,2899.0,NaN,1046.0,YokoSun,Подгузники-трусики,685
285,1698,158878851,0.30,NaN,901.0,NaN,23.0,NaN,1005.0,NaN,267.0,Starbucks,Кофе в капсулах,179
286,1698,439058771,0.00,NaN,3463.0,NaN,161.0,NaN,6274.0,NaN,1964.0,Ozon fresh,Сушки,617
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2703428,11183199,165179958,0.89,NaN,2531.0,NaN,297.0,1.0,1927.0,NaN,460.0,Тирольские Пироги,Пирог,13
2703429,11183199,1364982900,0.00,NaN,1001.0,NaN,104.0,NaN,2060.0,NaN,768.0,Полёт,Пряники,617
2703430,11183199,260759767,0.00,NaN,413.0,NaN,25.0,NaN,899.0,NaN,368.0,Полет,Печенье,829
2703431,11183199,1487783039,0.00,NaN,241.0,NaN,19.0,NaN,31.0,NaN,9.0,У Палыча,Печенье,923


In [54]:
candidates_df_pd['predict'] = model_load.predict(candidates_df_pd[cols], prediction_type="Probability")[:, 1]


/var/folders/3c/zb3vq_6960dbkm_ndj8dmrk40000gn/T/ipykernel_35617/3738339801.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  candidates_df_pd['predict'] = model_load.predict(candidates_df_pd[cols], prediction_type="Probability")[:, 1]


In [55]:
candidates_df_pd

,user_id,product_id,target,ui_num_click,i_num_click,ui_num_favorite,i_num_favorite,ui_num_to_cart,i_num_to_cart,ui_num_order,i_num_order,brand,type,category_id,predict
1,128,1562705601,0.00,NaN,20.0,NaN,NaN,NaN,1.0,NaN,NaN,YokoSun,Подгузники-трусики,685,0.408523
2,128,1023814617,0.00,2.0,8415.0,1.0,349.0,1.0,4693.0,1.0,1232.0,SNAQ FABRIQ,Спортивное питание,730,0.804605
3,128,149724906,0.89,NaN,2642.0,NaN,61.0,1.0,2899.0,NaN,1046.0,YokoSun,Подгузники-трусики,685,0.800541
285,1698,158878851,0.30,NaN,901.0,NaN,23.0,NaN,1005.0,NaN,267.0,Starbucks,Кофе в капсулах,179,0.433858
286,1698,439058771,0.00,NaN,3463.0,NaN,161.0,NaN,6274.0,NaN,1964.0,Ozon fresh,Сушки,617,0.465738
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2703428,11183199,165179958,0.89,NaN,2531.0,NaN,297.0,1.0,1927.0,NaN,460.0,Тирольские Пироги,Пирог,13,0.785135
2703429,11183199,1364982900,0.00,NaN,1001.0,NaN,104.0,NaN,2060.0,NaN,768.0,Полёт,Пряники,617,0.462949
2703430,11183199,260759767,0.00,NaN,413.0,NaN,25.0,NaN,899.0,NaN,368.0,Полет,Печенье,829,0.489839
2703431,11183199,1487783039,0.00,NaN,241.0,NaN,19.0,NaN,31.0,NaN,9.0,У Палыча,Печенье,923,0.389379


In [56]:
candidates_df_pd = (
    candidates_df_pd
    .sort_values(by=['predict'], ascending=False)
    .head(100)
)
candidates_df_pd['rank'] = np.arange(1, candidates_df_pd.shape[0] + 1)
candidates_df_pd['item_id'] = candidates_df_pd['product_id']

In [58]:
model_load.feature_names_

['ui_num_click',
 'i_num_click',
 'ui_num_favorite',
 'i_num_favorite',
 'ui_num_to_cart',
 'i_num_to_cart',
 'ui_num_order',
 'i_num_order']